# 04 - Hyperparameter Tuning With Optuna

This notebook runs a small hyperparameter tuning study on the saved training pseudobulks. It focuses on the architecture choices that matter most for the current MLP prototype: number of layers, number of neurons per layer, and dropout.

It uses Optuna's TPE sampler with pruning. That is a practical default for local deep-learning experiments: lighter than Ray, more efficient than a full grid, and easy to keep small.


In [1]:
from __future__ import annotations

import copy
import importlib.util
import json
import math
import random
from pathlib import Path
from typing import Sequence

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "optuna": "optuna",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". Install the project environment first, for example with `uv sync`."
    )

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root, where pyproject.toml is located.")

REFERENCE_LABEL = "MLPDeconv_mouse"

REFERENCE_DIR = PROJECT_ROOT / "data" / "references" / REFERENCE_LABEL
PSEUDOBULK_DIR = REFERENCE_DIR / "pseudobulk"
HPO_DIR = REFERENCE_DIR / "hyperparameter_tuning"
HPO_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# pin_memory speeds CPU-to-GPU batch transfer for DataLoaders when CUDA is available.
PIN_MEMORY = torch.cuda.is_available()
print(f"Device: {DEVICE}")



/home/sagemaker-user/user-default-efs/MLPDeconv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Load Saved Pseudobulks

The tuning study only uses the training pseudobulks. A validation split is created inside this notebook. The held-out test pseudobulks stay untouched for final evaluation in notebook 02.


In [2]:
def load_pseudobulk_dataset(npz_path: str | Path) -> dict:
    path = Path(npz_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing pseudobulk dataset: {path}. Run 01_create_pseudobulks.ipynb first.")
    with np.load(path, allow_pickle=True) as data:
        return {
            "X": data["X"].astype(np.float32),
            "y": data["y"].astype(np.float32),
            "genes": data["genes"].astype(str).tolist(),
            "cell_types": data["cell_types"].astype(str).tolist(),
            "sample_ids": data["sample_ids"].astype(str).tolist(),
        }


train_pseudobulk = load_pseudobulk_dataset(PSEUDOBULK_DIR / "train_pseudobulk.npz")
train_pb_X = train_pseudobulk["X"]
train_pb_y = train_pseudobulk["y"]
CELL_TYPES = train_pseudobulk["cell_types"]

print(f"Training pseudobulks: {train_pb_X.shape}")
print(f"Cell types: {len(CELL_TYPES)}")


Training pseudobulks: (20000, 23793)
Cell types: 15


## Preprocess Features


In [3]:
def normalize_bulk_counts(X: np.ndarray, scale_factor: float = 1_000_000.0) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    library_sizes = X.sum(axis=1, keepdims=True)
    if np.any(library_sizes <= 0):
        raise ValueError("At least one bulk sample has zero total counts and cannot be normalized.")
    normalized = (X / library_sizes) * scale_factor
    return np.log1p(normalized).astype(np.float32)


def split_train_validation(
    X: np.ndarray,
    y: np.ndarray,
    validation_fraction: float = 0.15,
    seed: int = RANDOM_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X.shape[0])
    n_validation = int(round(X.shape[0] * validation_fraction))
    n_validation = max(1, min(X.shape[0] - 1, n_validation))
    validation_idx = indices[:n_validation]
    train_idx = indices[n_validation:]
    return X[train_idx], y[train_idx], X[validation_idx], y[validation_idx]


def fit_standardizer(X_train: np.ndarray, eps: float = 1e-6) -> tuple[np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=0).astype(np.float32)
    std = X_train.std(axis=0).astype(np.float32)
    std[std < eps] = 1.0
    return mean, std


def apply_standardizer(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / std).astype(np.float32)


features_all = normalize_bulk_counts(train_pb_X)
X_train_raw, y_train, X_val_raw, y_val = split_train_validation(
    features_all,
    train_pb_y,
    validation_fraction=0.15,
    seed=RANDOM_SEED,
)
gene_mean, gene_std = fit_standardizer(X_train_raw)
X_train = apply_standardizer(X_train_raw, gene_mean, gene_std)
X_val = apply_standardizer(X_val_raw, gene_mean, gene_std)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")


X_train: (17000, 23793), y_train: (17000, 15)
X_val:   (3000, 23793), y_val:   (3000, 15)


## Model And Training Helpers


In [4]:
def set_reproducible_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class BulkFractionDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[index], self.y[index]


class MLPDeconvRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_sizes: Sequence[int],
        dropout: float,
    ):
        super().__init__()
        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.network(X), dim=1)


@torch.no_grad()
def predict_array(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device = DEVICE) -> np.ndarray:
    model.eval()
    tensor_dataset = torch.as_tensor(X, dtype=torch.float32)
    loader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
    predictions = []
    for X_batch in loader:
        predictions.append(model(X_batch.to(device)).cpu().numpy())
    return np.vstack(predictions).astype(np.float32)


def rmse_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(y_pred - y_true))))


def validation_rmse(model: nn.Module, X_val: np.ndarray, y_val: np.ndarray, device: torch.device = DEVICE) -> float:
    predictions = predict_array(model, X_val, device=device)
    return rmse_score(y_val, predictions)


## Define The Search Space

This search uses the following hyperparameters: number of layers, base width, dropout, learning rate, weight decay, and batch size. Hidden layers follow a pyramid pattern by default: each layer is half the previous width, with a minimum width of 64 neurons.


In [5]:
def build_hidden_sizes(n_layers: int, base_width: int, min_width: int = 64) -> tuple[int, ...]:
    return tuple(max(min_width, base_width // (2 ** layer_idx)) for layer_idx in range(n_layers))


N_TRIALS = 50
MAX_STEPS_PER_TRIAL = 1500
VALIDATION_EVERY = 100
PATIENCE = 6

print("Optuna will tune:")
print("- n_layers: 1 to 3")
print("- base_width: 128, 256, 512, 1024, 2048")
print("- dropout: 0.0 to 0.4")
print("- learning_rate: 1e-5 to 3e-3, log scale")
print("- weight_decay: 1e-7 to 1e-2, log scale")
print("- batch_size: 32, 64, 128, 256")
print(f"Optuna trials configured: {N_TRIALS}")


Optuna will tune:
- n_layers: 1 to 3
- base_width: 128, 256, 512, 1024, 2048
- dropout: 0.0 to 0.4
- learning_rate: 1e-5 to 3e-3, log scale
- weight_decay: 1e-7 to 1e-2, log scale
- batch_size: 32, 64, 128, 256
Optuna trials configured: 50


## Run Optuna Study

In [6]:
def objective(trial: optuna.Trial) -> float:
    n_layers = trial.suggest_int("n_layers", 1, 3)
    base_width = trial.suggest_categorical("base_width", [128, 256, 512, 1024, 2048])
    dropout = trial.suggest_float("dropout", 0.0, 0.4, step=0.1)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    hidden_sizes = build_hidden_sizes(n_layers=n_layers, base_width=base_width)
    trial.set_user_attr("hidden_sizes", list(hidden_sizes))

    set_reproducible_seed(RANDOM_SEED + trial.number)
    train_loader = DataLoader(
        BulkFractionDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )

    model = MLPDeconvRegressor(
        input_dim=X_train.shape[1],
        output_dim=y_train.shape[1],
        hidden_sizes=hidden_sizes,
        dropout=dropout,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_rmse = math.inf
    best_state = copy.deepcopy(model.state_dict())
    checks_without_improvement = 0
    global_step = 0
    stop_training = False

    while global_step < MAX_STEPS_PER_TRIAL and not stop_training:
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=PIN_MEMORY)
            y_batch = y_batch.to(DEVICE, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            global_step += 1

            should_validate = global_step == 1 or global_step % VALIDATION_EVERY == 0 or global_step == MAX_STEPS_PER_TRIAL
            if should_validate:
                val_rmse = validation_rmse(model, X_val, y_val, device=DEVICE)
                trial.report(val_rmse, step=global_step)

                if val_rmse < best_rmse:
                    best_rmse = val_rmse
                    best_state = copy.deepcopy(model.state_dict())
                    checks_without_improvement = 0
                else:
                    checks_without_improvement += 1

                if trial.should_prune():
                    raise optuna.TrialPruned()
                if checks_without_improvement >= PATIENCE:
                    stop_training = True
                    break
            if global_step >= MAX_STEPS_PER_TRIAL:
                break

    model.load_state_dict(best_state)
    return best_rmse


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=300, interval_steps=100)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best validation RMSE:", study.best_value)
print("Best params:", study.best_params)
print("Best hidden sizes:", study.best_trial.user_attrs["hidden_sizes"])

[I 2026-07-06 20:00:22,277] A new study created in memory with name: no-name-b5550bef-26b7-4ab8-aa3c-7dcc871bfa08
Best trial: 0. Best value: 0.0199781:   2%|▏         | 1/50 [00:15<12:38, 15.48s/it]

[I 2026-07-06 20:00:37,754] Trial 0 finished with value: 0.019978100433945656 and parameters: {'n_layers': 2, 'base_width': 128, 'dropout': 0.0, 'learning_rate': 0.0013983740016490966, 'weight_decay': 0.00010129197956845733, 'batch_size': 128}. Best is trial 0 with value: 0.019978100433945656.


Best trial: 1. Best value: 0.0140989:   4%|▍         | 2/50 [00:44<18:44, 23.43s/it]

[I 2026-07-06 20:01:06,745] Trial 1 finished with value: 0.01409890316426754 and parameters: {'n_layers': 1, 'base_width': 1024, 'dropout': 0.1, 'learning_rate': 0.0003278187653397617, 'weight_decay': 4.982752357076448e-07, 'batch_size': 256}. Best is trial 1 with value: 0.01409890316426754.


Best trial: 1. Best value: 0.0140989:   6%|▌         | 3/50 [00:52<12:42, 16.23s/it]

...

[I 2026-07-06 20:13:41,632] Trial 47 pruned. 


Best trial: 1. Best value: 0.0140989:  98%|█████████▊| 49/50 [13:22<00:10, 10.97s/it]

[I 2026-07-06 20:13:44,562] Trial 48 pruned. 


Best trial: 1. Best value: 0.0140989: 100%|██████████| 50/50 [13:26<00:00, 16.12s/it]

[I 2026-07-06 20:13:48,300] Trial 49 pruned. 
Best validation RMSE: 0.01409890316426754
Best params: {'n_layers': 1, 'base_width': 1024, 'dropout': 0.1, 'learning_rate': 0.0003278187653397617, 'weight_decay': 4.982752357076448e-07, 'batch_size': 256}
Best hidden sizes: [1024]


## Save Tuning Results


In [7]:
trial_rows = []
for trial in study.trials:
    trial_rows.append(
        {
            "trial": trial.number,
            "state": trial.state.name,
            "val_rmse": trial.value,
            "n_layers": trial.params.get("n_layers"),
            "base_width": trial.params.get("base_width"),
            "hidden_sizes": trial.user_attrs.get("hidden_sizes"),
            "dropout": trial.params.get("dropout"),
            "learning_rate": trial.params.get("learning_rate"),
            "weight_decay": trial.params.get("weight_decay"),
            "batch_size": trial.params.get("batch_size"),
        }
    )

trials_df = pd.DataFrame(trial_rows)
trials_path = HPO_DIR / "optuna_trials.csv"
best_config_path = HPO_DIR / "best_hyperparameters.json"
plot_path = HPO_DIR / "optuna_validation_rmse_by_trial.png"

trials_df.to_csv(trials_path, index=False)
best_config = {
    "objective": "validation_rmse",
    "best_value": study.best_value,
    "n_layers": study.best_params["n_layers"],
    "base_width": study.best_params["base_width"],
    "hidden_sizes": study.best_trial.user_attrs["hidden_sizes"],
    "dropout": study.best_params["dropout"],
    "learning_rate": study.best_params["learning_rate"],
    "weight_decay": study.best_params["weight_decay"],
    "batch_size": study.best_params["batch_size"],
    "fixed_training_parameters": {
        "max_steps_per_trial": MAX_STEPS_PER_TRIAL,
        "validation_every": VALIDATION_EVERY,
        "patience": PATIENCE,
        "loss": "MSE",
    },
}
best_config_path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")

completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(completed["trial"], completed["val_rmse"], marker="o", linewidth=1.5)
ax.set_xlabel("Trial")
ax.set_ylabel("Validation RMSE")
ax.set_title("Optuna architecture and training-parameter search")
fig.tight_layout()
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.close(fig)

display(trials_df.sort_values("val_rmse", na_position="last"))
print(f"Saved trials to {trials_path}")
print(f"Saved best config to {best_config_path}")
print(f"Saved tuning plot to {plot_path}")
best_config


,trial,state,val_rmse,n_layers,base_width,hidden_sizes,dropout,learning_rate,weight_decay,batch_size
1,1,COMPLETE,0.014099,1,1024,[1024],0.1,0.000328,4.982752e-07,256
44,44,COMPLETE,0.014991,2,2048,"[2048, 1024]",0.4,0.000060,1.858183e-06,128
34,34,COMPLETE,0.015028,2,2048,"[2048, 1024]",0.4,0.000059,1.895031e-07,128
20,20,COMPLETE,0.015082,2,2048,"[2048, 1024]",0.4,0.000048,5.465302e-07,128
33,33,COMPLETE,0.015130,2,2048,"[2048, 1024]",0.4,0.000046,2.415847e-07,128
21,21,COMPLETE,0.015166,2,2048,"[2048, 1024]",0.4,0.000044,5.304553e-07,128
42,42,COMPLETE,0.015187,2,2048,"[2048, 1024]",0.4,0.000041,3.470894e-07,128
22,22,COMPLETE,0.015284,2,2048,"[2048, 1024]",0.4,0.000031,4.997008e-07,128
43,43,PRUNED,0.015352,2,2048,"[2048, 1024]",0.4,0.000032,1.132984e-03,128
3,3,COMPLETE,0.015507,2,1024,"[1024, 512]",0.3,0.000059,3.984191e-05,128


Saved trials to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/MLPDeconv_mouse/hyperparameter_tuning/optuna_trials.csv
Saved best config to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/MLPDeconv_mouse/hyperparameter_tuning/best_hyperparameters.json
Saved tuning plot to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/MLPDeconv_mouse/hyperparameter_tuning/optuna_validation_rmse_by_trial.png


{'objective': 'validation_rmse',
 'best_value': 0.01409890316426754,
 'n_layers': 1,
 'base_width': 1024,
 'hidden_sizes': [1024],
 'dropout': 0.1,
 'learning_rate': 0.0003278187653397617,
 'weight_decay': 4.982752357076448e-07,
 'batch_size': 256,
 'fixed_training_parameters': {'max_steps_per_trial': 1500,
  'validation_every': 100,
  'patience': 6,
  'loss': 'MSE'}}

## Use The Best Configuration

Copy the `hidden_sizes` and `dropout` from `best_hyperparameters.json` into `TrainingConfig` in `02_train_eval_model.ipynb`, then rerun notebook 02 for the final model and held-out test evaluation.


# Repeat for the Sub Celltype level



In [8]:
# Parameters

REFERENCE_LABEL = "mouse_sub"

ARTIFACT_DIR = PROJECT_ROOT / "data" / "artifacts"
PSEUDOBULK_DIR = ARTIFACT_DIR / "pseudobulk" / REFERENCE_LABEL
HPO_DIR = ARTIFACT_DIR / "hyperparameter_tuning" / REFERENCE_LABEL
HPO_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
# Load Pseudobulk

train_pseudobulk = load_pseudobulk_dataset(PSEUDOBULK_DIR / "train_pseudobulk.npz")
train_pb_X = train_pseudobulk["X"]
train_pb_y = train_pseudobulk["y"]
CELL_TYPES = train_pseudobulk["cell_types"]

print(f"Training pseudobulks: {train_pb_X.shape}")
print(f"Cell types: {len(CELL_TYPES)}")

# Preprocess Features
features_all = normalize_bulk_counts(train_pb_X)
X_train_raw, y_train, X_val_raw, y_val = split_train_validation(
    features_all,
    train_pb_y,
    validation_fraction=0.15,
    seed=RANDOM_SEED,
)
gene_mean, gene_std = fit_standardizer(X_train_raw)
X_train = apply_standardizer(X_train_raw, gene_mean, gene_std)
X_val = apply_standardizer(X_val_raw, gene_mean, gene_std)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")

FileNotFoundError: Missing pseudobulk dataset: /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/artifacts/pseudobulk/mouse_sub/train_pseudobulk.npz. Run 01_create_pseudobulks.ipynb first.

In [ ]:
# Run previously defined Optuna study

def objective(trial: optuna.Trial) -> float:
    n_layers = trial.suggest_int("n_layers", 1, 3)
    base_width = trial.suggest_categorical("base_width", [128, 256, 512, 1024, 2048])
    dropout = trial.suggest_float("dropout", 0.0, 0.4, step=0.1)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    hidden_sizes = build_hidden_sizes(n_layers=n_layers, base_width=base_width)
    trial.set_user_attr("hidden_sizes", list(hidden_sizes))

    set_reproducible_seed(RANDOM_SEED + trial.number)
    train_loader = DataLoader(
        BulkFractionDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )

    model = MLPDeconvRegressor(
        input_dim=X_train.shape[1],
        output_dim=y_train.shape[1],
        hidden_sizes=hidden_sizes,
        dropout=dropout,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.L1Loss()

    best_rmse = math.inf
    best_state = copy.deepcopy(model.state_dict())
    checks_without_improvement = 0
    global_step = 0
    stop_training = False

    while global_step < MAX_STEPS_PER_TRIAL and not stop_training:
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=PIN_MEMORY)
            y_batch = y_batch.to(DEVICE, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            global_step += 1

            should_validate = global_step == 1 or global_step % VALIDATION_EVERY == 0 or global_step == MAX_STEPS_PER_TRIAL
            if should_validate:
                val_rmse = validation_rmse(model, X_val, y_val, device=DEVICE)
                trial.report(val_rmse, step=global_step)

                if val_rmse < best_rmse:
                    best_rmse = val_rmse
                    best_state = copy.deepcopy(model.state_dict())
                    checks_without_improvement = 0
                else:
                    checks_without_improvement += 1

                if trial.should_prune():
                    raise optuna.TrialPruned()
                if checks_without_improvement >= PATIENCE:
                    stop_training = True
                    break
            if global_step >= MAX_STEPS_PER_TRIAL:
                break

    model.load_state_dict(best_state)
    return best_rmse

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=300, interval_steps=100)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best validation RMSE:", study.best_value)
print("Best params:", study.best_params)
print("Best hidden sizes:", study.best_trial.user_attrs["hidden_sizes"])

[I 2026-06-01 15:16:11,591] A new study created in memory with name: no-name-5914b99e-fd67-444d-964d-a0cac378e2f0
Best trial: 0. Best value: 0.0189985:   2%|▏         | 1/50 [00:05<04:10,  5.10s/it]

[I 2026-06-01 15:16:16,693] Trial 0 finished with value: 0.018998483195900917 and parameters: {'n_layers': 2, 'base_width': 128, 'dropout': 0.0, 'learning_rate': 0.0013983740016490966, 'weight_decay': 0.00010129197956845733, 'batch_size': 128}. Best is trial 0 with value: 0.018998483195900917.


Best trial: 1. Best value: 0.0132428:   4%|▍         | 2/50 [00:17<07:18,  9.14s/it]

[I 2026-06-01 15:16:28,665] Trial 1 finished with value: 0.013242784887552261 and parameters: {'n_layers': 1, 'base_width': 1024, 'dropout': 0.1, 'learning_rate': 0.0003278187653397617, 'weight_decay': 4.982752357076448e-07, 'batch_size': 256}. Best is trial 1 with value: 0.013242784887552261.


Best trial: 1. Best value: 0.0132428:   6%|▌         | 3/50 [00:20<04:59,  6.37s/it]

...

[I 2026-06-01 15:19:36,510] Trial 47 pruned. 


Best trial: 1. Best value: 0.0132428:  98%|█████████▊| 49/50 [03:30<00:05,  5.36s/it]

[I 2026-06-01 15:19:42,199] Trial 48 pruned. 


Best trial: 1. Best value: 0.0132428: 100%|██████████| 50/50 [03:32<00:00,  4.25s/it]

[I 2026-06-01 15:19:44,085] Trial 49 pruned. 
Best validation RMSE: 0.013242784887552261
Best params: {'n_layers': 1, 'base_width': 1024, 'dropout': 0.1, 'learning_rate': 0.0003278187653397617, 'weight_decay': 4.982752357076448e-07, 'batch_size': 256}
Best hidden sizes: [1024]


In [ ]:
# Save tunning results

trial_rows = []
for trial in study.trials:
    trial_rows.append(
        {
            "trial": trial.number,
            "state": trial.state.name,
            "val_rmse": trial.value,
            "n_layers": trial.params.get("n_layers"),
            "base_width": trial.params.get("base_width"),
            "hidden_sizes": trial.user_attrs.get("hidden_sizes"),
            "dropout": trial.params.get("dropout"),
            "learning_rate": trial.params.get("learning_rate"),
            "weight_decay": trial.params.get("weight_decay"),
            "batch_size": trial.params.get("batch_size"),
        }
    )

trials_df = pd.DataFrame(trial_rows)
trials_path = HPO_DIR / "optuna_trials.csv"
best_config_path = HPO_DIR / "best_hyperparameters.json"
plot_path = HPO_DIR / "optuna_validation_rmse_by_trial.png"

trials_df.to_csv(trials_path, index=False)
best_config = {
    "objective": "validation_rmse",
    "best_value": study.best_value,
    "n_layers": study.best_params["n_layers"],
    "base_width": study.best_params["base_width"],
    "hidden_sizes": study.best_trial.user_attrs["hidden_sizes"],
    "dropout": study.best_params["dropout"],
    "learning_rate": study.best_params["learning_rate"],
    "weight_decay": study.best_params["weight_decay"],
    "batch_size": study.best_params["batch_size"],
    "fixed_training_parameters": {
        "max_steps_per_trial": MAX_STEPS_PER_TRIAL,
        "validation_every": VALIDATION_EVERY,
        "patience": PATIENCE,
        "loss": "MSELoss",
    },
}
best_config_path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")

completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(completed["trial"], completed["val_rmse"], marker="o", linewidth=1.5)
ax.set_xlabel("Trial")
ax.set_ylabel("Validation RMSE")
ax.set_title("Optuna architecture and training-parameter search")
fig.tight_layout()
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.close(fig)

display(trials_df.sort_values("val_rmse", na_position="last"))
print(f"Saved trials to {trials_path}")
print(f"Saved best config to {best_config_path}")
print(f"Saved tuning plot to {plot_path}")
best_config


,trial,state,val_rmse,n_layers,base_width,hidden_sizes,dropout,learning_rate,weight_decay,batch_size
1,1,COMPLETE,0.013243,1,1024,[1024],0.1,0.000328,4.982752e-07,256
37,37,COMPLETE,0.014160,3,2048,"[2048, 1024, 512]",0.0,0.000024,3.673860e-04,64
40,40,COMPLETE,0.014598,3,2048,"[2048, 1024, 512]",0.0,0.000030,1.026381e-07,64
4,4,COMPLETE,0.015775,3,512,"[512, 256, 128]",0.0,0.000064,8.777816e-06,64
41,41,PRUNED,0.015867,3,2048,"[2048, 1024, 512]",0.0,0.000026,1.302287e-07,64
47,47,PRUNED,0.015934,3,512,"[512, 256, 128]",0.0,0.000045,4.563255e-07,256
3,3,COMPLETE,0.016506,2,1024,"[1024, 512]",0.3,0.000059,3.984191e-05,128
42,42,PRUNED,0.016770,3,2048,"[2048, 1024, 512]",0.0,0.000017,3.510635e-07,64
43,43,PRUNED,0.017207,3,2048,"[2048, 1024, 512]",0.0,0.000038,2.175187e-07,64
5,5,PRUNED,0.017302,2,1024,"[1024, 512]",0.0,0.000010,1.194833e-03,128


Saved trials to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/artifacts/hyperparameter_tuning/mouse_sub/optuna_trials.csv
Saved best config to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/artifacts/hyperparameter_tuning/mouse_sub/best_hyperparameters.json
Saved tuning plot to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/artifacts/hyperparameter_tuning/mouse_sub/optuna_validation_rmse_by_trial.png


{'objective': 'validation_rmse',
 'best_value': 0.013242784887552261,
 'n_layers': 1,
 'base_width': 1024,
 'hidden_sizes': [1024],
 'dropout': 0.1,
 'learning_rate': 0.0003278187653397617,
 'weight_decay': 4.982752357076448e-07,
 'batch_size': 256,
 'fixed_training_parameters': {'max_steps_per_trial': 1500,
  'validation_every': 100,
  'patience': 6,
  'loss': 'MSELoss'}}